# VGGT (Meta)

Relative depth. Multi-view transformer used in single-image mode.

**Runtime:** GPU (L4 or A100)

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, random, time, gc, sys, json, importlib, subprocess
import numpy as np
import cv2
import torch
from PIL import Image

DATASET_IMAGES = '/content/drive/MyDrive/Corn Seed Dataset/test/images'
DATASET_LABELS = '/content/drive/MyDrive/Corn Seed Dataset/test/labels'
SAVE_DIR       = '/content/drive/MyDrive/Corn Seed Dataset/depth_comparison_outputs'
SAMPLE_FILE    = os.path.join(SAVE_DIR, 'sample_images.txt')

assert os.path.isdir(DATASET_IMAGES), f'Dataset not found: {DATASET_IMAGES}'
os.makedirs(SAVE_DIR, exist_ok=True)

def save_depth(depth_np, stem, model_name):
    d = np.array(depth_np, dtype=np.float32)
    while d.ndim > 2: d = d[0]
    d_norm = (d - d.min()) / (d.max() - d.min() + 1e-8)
    for sub, img in [('depth', (d_norm*65535).astype(np.uint16)),
                     ('vis',   cv2.applyColorMap((d_norm*255).astype(np.uint8), cv2.COLORMAP_INFERNO))]:
        p = os.path.join(SAVE_DIR, model_name, sub)
        os.makedirs(p, exist_ok=True)
        cv2.imwrite(os.path.join(p, f'{stem}.png'), img)

def clear_gpu():
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

print('Setup done.')
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

# Load shared sample list from Drive (run depth_pro.ipynb first)
assert os.path.exists(SAMPLE_FILE), f'Run depth_pro.ipynb first to create sample_images.txt'
SAMPLES = [l.strip() for l in open(SAMPLE_FILE) if l.strip()]
print(f'Loaded {len(SAMPLES)} samples')

## Install + run

In [ ]:
!git clone https://github.com/facebookresearch/vggt /content/vggt 2>/dev/null || true
!grep -v '^torch==' /content/vggt/requirements.txt | pip install -q -r /dev/stdin

sys.path.insert(0, '/content/vggt')
importlib.invalidate_caches()
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images

MODEL_NAME = 'vggt'
dtype = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
model = VGGT.from_pretrained('facebook/VGGT-1B').cuda().eval()

depth_key = None
times = []
for img_name in SAMPLES:
    stem = img_name.split('.')[0]
    images = load_and_preprocess_images([os.path.join(DATASET_IMAGES, img_name)]).cuda()
    t0 = time.time()
    with torch.no_grad():
        with torch.cuda.amp.autocast(dtype=dtype):
            predictions = model(images)
    torch.cuda.synchronize(); times.append(time.time()-t0)
    if depth_key is None:
        depth_key = next((k for k in ['depth_map', 'depth'] if k in predictions), None)
        if depth_key is None:
            print('ERROR: no depth key found. Keys:', list(predictions.keys())); break
        print(f'Using depth key: {depth_key!r}')
    d = predictions[depth_key].cpu().float().numpy()
    while d.ndim > 2: d = d[0]
    save_depth(d, stem, MODEL_NAME)

print(f'Done -- {len(times)} images, avg {np.mean(times):.3f}s/img')
del model; sys.path.remove('/content/vggt'); clear_gpu()

## Confirm saved

In [ ]:
model_dir = os.path.join(SAVE_DIR, MODEL_NAME)
n_depth = len(os.listdir(os.path.join(model_dir, 'depth')))
n_vis   = len(os.listdir(os.path.join(model_dir, 'vis')))
print(f'{MODEL_NAME}: {n_depth} depth maps, {n_vis} visualizations saved to Drive')